# 02. Training, Evaluation, Ablations & TFLite Export

Trains the NoNooze custom from-scratch 4-block CNN (M1) and baseline models:
- Person-level train/validation/test splits (Zero subject leakage)
- Augmentation pipeline (CLAHE, gamma, noise, rotation)
- False-Open Rate (FAR) optimization
- Temperature scaling calibration
- Float16 & Int8 quantization and export

In [ ]:
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image

from ml.src.dataset import load_dataset_from_directory, split_by_subject, verify_no_subject_leakage
from ml.src.models import build_m1_eye_cnn, build_m2_mobilenet, build_m3_separable
from ml.src.augmentations import EyeAugmentor
from ml.src.metrics import compute_classification_metrics, TemperatureScaler
from ml.src.export import export_to_tflite, verify_tflite_parity

print("TensorFlow Version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 1. Load and Partition Data with Person-Level Invariant
data_dir = Path("ml/mock_data")
all_samples = load_dataset_from_directory(data_dir)

# Designate held-out test subjects (never seen during training)
mrl_test_subs = {"s0009", "s0010"}
phone_test_subs = {"phone_user_05"}
test_subject_ids = mrl_test_subs.union(phone_test_subs)

train_samples, val_samples, test_samples = split_by_subject(
    all_samples, test_subject_ids=test_subject_ids, val_ratio=0.15, seed=42
)

verify_no_subject_leakage(train_samples, val_samples, test_samples)
print(f"Partitions: {len(train_samples)} Train | {len(val_samples)} Val | {len(test_samples)} Test")

In [ ]:
# 2. Load arrays and apply augmentations
augmentor = EyeAugmentor(rng_seed=42)

def samples_to_numpy(sample_list, augment=False):
    X, y = [], []
    for s in sample_list:
        img = Image.open(s.path).convert("L").resize((48, 48))
        arr = np.array(img, dtype=np.float32) / 255.0
        if augment:
            arr = augmentor(arr)
        X.append(np.expand_dims(arr, axis=-1))
        y.append(s.label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32).reshape(-1, 1)

X_train, y_train = samples_to_numpy(train_samples, augment=True)
X_val, y_val = samples_to_numpy(val_samples, augment=False)
X_test, y_test = samples_to_numpy(test_samples, augment=False)

print(f"X_train shape: {X_train.shape}, Class balance: {np.mean(y_train):.2f} open")

In [ ]:
# 3. Build & Train Model M1 (From-Scratch 4-Block CNN)
m1_model = build_m1_eye_cnn(input_shape=(48, 48, 1))
print(f"M1 Parameter Count: {m1_model.count_params():,} (< 100k budget)")

m1_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = m1_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16,
    verbose=1
)

In [ ]:
# 4. Evaluate Metrics on Held-out Test Sets (T1, T2, T3)
y_pred_prob = m1_model.predict(X_test).flatten()
metrics = compute_classification_metrics(y_test.flatten(), y_pred_prob, threshold=0.5)

print("=== Held-Out Test Evaluation ===")
print(f"Accuracy:         {metrics['accuracy']*100:.2f}%")
print(f"ROC-AUC:          {metrics['roc_auc']:.4f}")
print(f"F1 Score:         {metrics['f1']:.4f}")
print(f"False-Open Rate:  {metrics['false_open_rate']*100:.2f}% (Critical Alarm Metric)")
print(f"False-Closed Rate:{metrics['false_closed_rate']*100:.2f}%")

In [ ]:
# 5. Export to TFLite (Float16) and Verify Parity
out_tflite = Path("assets/models/eye_cnn_v1.tflite")
export_meta = export_to_tflite(m1_model, out_tflite, quantization="float16")
print(f"Exported TFLite Size: {export_meta['size_kb']:.2f} KB (< 400 KB budget)")

parity = verify_tflite_parity(m1_model, out_tflite, X_test[:10])
print(f"Parity Max Absolute Diff: {parity['max_abs_diff']:.6f} (Passed: {parity['is_parity_ok']})")